<a href="https://colab.research.google.com/github/tisiflavio/uselectionanalysis/blob/main/FLAVIO_TISI_us_election_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗳️ US Presidential Elections: 2016 vs 2020 vs 2024
### A Swing State & County-Level Analysis with Demographic Context

**What this notebook explores:**
- How swing state margins shifted across three election cycles
- County-level vote share changes (Republican vs Democrat swings)
- Demographic correlations: education, income, and race vs vote share

**Data sources used:**
- [tonmcg/US_County_Level_Election_Results_08-24](https://github.com/tonmcg/US_County_Level_Election_Results_08-24) — 2016, 2020, 2024 results
- MIT Election Lab

> **Note for beginners:** Each section starts with a plain-English explanation of what the code does before you run it.

---
## 0. Setup — Install & Import Libraries

Run this cell first. It installs any missing packages and imports everything we need.

In [ ]:
# Install required packages (only needed once)
!pip install pandas plotly nbformat requests --quiet

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ All libraries loaded successfully!')

✅ All libraries loaded successfully!


---
## 1. Load the Data

We load county-level election results for 2016, 2020, and 2024. Each row is one county in one year, with columns for Republican and Democrat vote totals.

We use a well-maintained GitHub dataset that combines all three cycles.

In [ ]:
# ── via GitHub mirror ──────────────────────
url_2016_2020 = (
    "https://raw.githubusercontent.com/tonmcg/US_County_Level_Election_Results_08-24"
    "/master/2020_US_County_Level_Presidential_Results.csv"
)
url_2016 = (
    "https://raw.githubusercontent.com/tonmcg/US_County_Level_Election_Results_08-24"
    "/master/2016_US_County_Level_Presidential_Results.csv"
)
url_2024 = (
    "https://raw.githubusercontent.com/tonmcg/US_County_Level_Election_Results_08-24"
    "/master/2024_US_County_Level_Presidential_Results.csv"
)

df_2020_raw = pd.read_csv(url_2016_2020)
df_2016_raw = pd.read_csv(url_2016)
df_2024_raw = pd.read_csv(url_2024)

print("2016 shape:", df_2016_raw.shape)
print("2020 shape:", df_2020_raw.shape)
print("2024 shape:", df_2024_raw.shape)

df_2024_raw.head(3)

2016 shape: (3141, 11)
2020 shape: (3152, 10)
2024 shape: (3160, 10)


,state_name,county_fips,county_name,votes_gop,votes_dem,total_votes,diff,per_gop,per_dem,per_point_diff
0,Alabama,1001,Autauga County,20484,7439,28190,13045.00,0.73,0.26,0.46
1,Alabama,1003,Baldwin County,95798,24934,121808,70864.00,0.79,0.20,0.58
2,Alabama,1005,Barbour County,5606,4158,9832,1448.00,0.57,0.42,0.15


---
## 2. Clean & Standardize

Each dataset has slightly different column names. We rename them to a consistent schema and add a `year` column so we can stack them into one big DataFrame later.



In [ ]:
def standardize(df, year):
    """
    Normalize column names across datasets and compute derived columns.
    Returns a clean DataFrame with consistent schema.
    """
    # Print original columns so you can see what we're working with
    print(f"\n{year} original columns:", df.columns.tolist())

    # ── Map each year's column names to a standard set ────────────────────
    col_maps = {
        2016: {
            'combined_fips': 'fips',
            'county_name':   'county',
            'state_abbr':    'state',
            'votes_dem':     'dem_votes',
            'votes_gop':     'rep_votes',
            'total_votes':   'total_votes',
        },
        2020: {
            'county_fips':   'fips',
            'county_name':   'county',
            'state_name':    'state',
            'votes_dem':     'dem_votes',
            'votes_gop':     'rep_votes',
            'total_votes':   'total_votes',
        },
        2024: {
            'county_fips':   'fips',
            'county_name':   'county',
            'state_name':    'state',
            'votes_dem':     'dem_votes',
            'votes_gop':     'rep_votes',
            'total_votes':   'total_votes',
        },
    }

    df = df.rename(columns=col_maps[year])
    keep = ['fips', 'county', 'state', 'dem_votes', 'rep_votes', 'total_votes']
    df = df[[c for c in keep if c in df.columns]].copy()

    # Convert state abbreviations to full names for 2016 data to ensure consistency
    if year == 2016:
        state_abbr_to_full = {
            'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
            'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'DC': 'District of Columbia',
            'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois',
            'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana',
            'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
            'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
            'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
            'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon',
            'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota',
            'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia',
            'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
        }
        df['state'] = df['state'].replace(state_abbr_to_full)

    # ── Derived columns ───────────────────────────────────────────────────
    df['year']        = year
    df['dem_pct']     = df['dem_votes'] / df['total_votes'] * 100
    df['rep_pct']     = df['rep_votes'] / df['total_votes'] * 100
    # Positive = Republican advantage; negative = Democrat advantage
    df['margin']      = df['rep_pct'] - df['dem_pct']
    df['winner']      = np.where(df['dem_votes'] > df['rep_votes'], 'Democrat', 'Republican')
    df['fips']        = df['fips'].astype(str).str.zfill(5)  # ensure 5-digit FIPS code

    # Drop rows missing critical data
    df = df.dropna(subset=['dem_votes', 'rep_votes', 'total_votes'])
    df = df[df['total_votes'] > 0]

    print(f"{year} cleaned shape: {df.shape}")
    return df

df_2016 = standardize(df_2016_raw, 2016)
df_2020 = standardize(df_2020_raw, 2020)
df_2024 = standardize(df_2024_raw, 2024)

# Stack all three years into one DataFrame
df_all = pd.concat([df_2016, df_2020, df_2024], ignore_index=True)
print("\n✅ Combined shape:", df_all.shape)
df_all.head()

print(df_all)


2016 original columns: ['Unnamed: 0', 'votes_dem', 'votes_gop', 'total_votes', 'per_dem', 'per_gop', 'diff', 'per_point_diff', 'state_abbr', 'county_name', 'combined_fips']
2016 cleaned shape: (3141, 11)

2020 original columns: ['state_name', 'county_fips', 'county_name', 'votes_gop', 'votes_dem', 'total_votes', 'diff', 'per_gop', 'per_dem', 'per_point_diff']
2020 cleaned shape: (3152, 11)

2024 original columns: ['state_name', 'county_fips', 'county_name', 'votes_gop', 'votes_dem', 'total_votes', 'diff', 'per_gop', 'per_dem', 'per_point_diff']
2024 cleaned shape: (3160, 11)

✅ Combined shape: (9453, 11)
       fips             county    state  dem_votes  rep_votes  total_votes  \
0     02013             Alaska   Alaska   93003.00  130413.00    246588.00   
1     02016             Alaska   Alaska   93003.00  130413.00    246588.00   
2     02020             Alaska   Alaska   93003.00  130413.00    246588.00   
3     02050             Alaska   Alaska   93003.00  130413.00    246588.00 

---
## 3. Define Swing States

We focus our analysis on the 7 key battleground states that determined all three elections.

In [ ]:
SWING_STATES = [
    'Pennsylvania', 'Michigan', 'Wisconsin',
    'Arizona', 'Georgia', 'Nevada', 'North Carolina'
]

# Filter to swing states only
# Note: state column may be abbreviation or full name depending on the year
# We'll normalize — check what we have first
print("Sample state values:", df_all['state'].unique()[:10])

df_swing = df_all[df_all['state'].isin(SWING_STATES)].copy()
print(f"\nSwing state rows: {len(df_swing)} across {df_swing['state'].nunique()} states")

Sample state values: ['Alaska' 'Alabama' 'Arkansas' 'Arizona' 'California' 'Colorado'
 'Connecticut' 'District of Columbia' 'Delaware' 'Florida']

Swing state rows: 1539 across 7 states


---
## 4. State-Level Margins Across Three Elections

**What we're doing:** Group by state and year, sum all county votes, then compute the overall margin. A positive margin = Republican win; negative = Democrat win.



In [ ]:
# Aggregate to state level

state_totals = (
    df_swing
    .groupby(['state', 'year'])
    .agg(
        dem_votes   = ('dem_votes',   'sum'),
        rep_votes   = ('rep_votes',   'sum'),
        total_votes = ('total_votes', 'sum')
    )
    .reset_index()
)

state_totals['dem_pct'] = state_totals['dem_votes'] / state_totals['total_votes'] * 100
state_totals['rep_pct'] = state_totals['rep_votes'] / state_totals['total_votes'] * 100
state_totals['margin']  = state_totals['rep_pct'] - state_totals['dem_pct']
state_totals['winner']  = np.where(state_totals['margin'] > 0, 'Republican', 'Democrat')

state_totals.head(10)

Adesso la prova
Fine prova


,state,year,dem_votes,rep_votes,total_votes,dem_pct,rep_pct,margin,winner
0,Arizona,2016,936250.00,1021154.00,2062810.00,45.39,49.50,4.12,Republican
1,Arizona,2020,1672143.00,1661686.00,3387326.00,49.36,49.06,-0.31,Democrat
2,Arizona,2024,1582860.00,1770242.00,3389319.00,46.70,52.23,5.53,Republican
3,Georgia,2016,1837300.00,2068623.00,4029564.00,45.60,51.34,5.74,Republican
4,Georgia,2020,2473633.00,2461854.00,4997716.00,49.50,49.26,-0.24,Democrat
5,Georgia,2024,2548017.00,2663117.00,5250047.00,48.53,50.73,2.19,Republican
6,Michigan,2016,2268193.00,2279805.00,4790917.00,47.34,47.59,0.24,Republican
7,Michigan,2020,2804040.00,2649852.00,5539302.00,50.62,47.84,-2.78,Democrat
8,Michigan,2024,2736533.00,2816636.00,5662504.00,48.33,49.74,1.41,Republican
9,Nevada,2016,537753.00,511319.00,1122990.00,47.89,45.53,-2.35,Democrat


In [ ]:
# ── Chart: Swing state margins by year ────────────────────────────────────
fig = px.bar(
    state_totals,
    x='margin',
    y='state',
    color='winner',
    facet_col='year',
    orientation='h',
    color_discrete_map={'Republican': '#E63946', 'Democrat': '#457B9D'},
    title='Swing State Margins: 2016 vs 2020 vs 2024',
    labels={'margin': 'Margin (R − D, %)', 'state': ''},
    hover_data={'dem_pct': ':.1f', 'rep_pct': ':.1f'}
)
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_layout(height=420, showlegend=True)
fig.show()

---
## 5. Margin Shift: How Much Did Each State Move?

We pivot the data so each row is a state, with columns for each year's margin. Then we compute the shift from 2016→2024.



In [ ]:
# Pivot: rows = states, columns = years
margin_pivot = state_totals.pivot_table(
    index='state',
    columns='year',
    values='margin'
).round(2)

margin_pivot.columns.name = None  # clean up column label

# Compute shifts
margin_pivot['shift_16_to_20'] = margin_pivot[2020] - margin_pivot[2016]
margin_pivot['shift_20_to_24'] = margin_pivot[2024] - margin_pivot[2020]
margin_pivot['shift_16_to_24'] = margin_pivot[2024] - margin_pivot[2016]

# Positive shift = moved toward Republicans; negative = toward Democrats
margin_pivot.sort_values('shift_16_to_24', ascending=False)

,2016,2020,2024,shift_16_to_20,shift_20_to_24,shift_16_to_24
state,,,,,,
Nevada,-2.35,-2.39,3.10,-0.04,5.49,5.45
Arizona,4.12,-0.31,5.53,-4.43,5.84,1.41
Michigan,0.24,-2.78,1.41,-3.02,4.19,1.17
Pennsylvania,1.14,-1.18,1.71,-2.32,2.89,0.57
Wisconsin,0.78,-0.62,0.86,-1.40,1.48,0.08
North Carolina,3.83,1.35,3.22,-2.48,1.87,-0.61
Georgia,5.74,-0.24,2.19,-5.98,2.43,-3.55


In [ ]:
# ── Chart: Net shift 2016 → 2024 ──────────────────────────────────────────
shift_df = margin_pivot[['shift_16_to_24']].reset_index()
shift_df['direction'] = np.where(shift_df['shift_16_to_24'] > 0, 'Toward Republican', 'Toward Democrat')

fig = px.bar(
    shift_df.sort_values('shift_16_to_24'),
    x='shift_16_to_24',
    y='state',
    color='direction',
    orientation='h',
    color_discrete_map={'Toward Republican': '#E63946', 'Toward Democrat': '#457B9D'},
    title='Net Margin Shift in Swing States: 2016 → 2024 (pp)',
    labels={'shift_16_to_24': 'Shift in margin (percentage points)', 'state': ''}
)
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_layout(height=380)
fig.show()

---
## 6. County-Level Analysis: Flipped Counties

A **flipped county** is one that voted differently in 2024 vs 2020 (or 2016). These are the most politically significant data points.



In [ ]:
# Prepare county-level winner per year
def county_winners(df, year):
    return (
        df[df['year'] == year][['fips', 'county', 'state', 'winner', 'margin']]
        .rename(columns={'winner': f'winner_{year}', 'margin': f'margin_{year}'})
    )

w2016 = county_winners(df_swing, 2016)
w2020 = county_winners(df_swing, 2020)
w2024 = county_winners(df_swing, 2024)

# Merge all three on FIPS code (unique county identifier)
flips = (
    w2016
    .merge(w2020, on=['fips', 'county', 'state'])
    .merge(w2024, on=['fips', 'county', 'state'])
)

# Tag flip type
def flip_label(row):
    w16, w20, w24 = row['winner_2016'], row['winner_2020'], row['winner_2024']
    if w20 != w16 and w24 == w20:  return 'Flipped in 2020, held'
    if w24 != w20 and w20 == w16:  return 'Flipped in 2024'
    if w24 != w20 and w20 != w16:  return 'Flipped twice'
    return 'Consistent'

flips['flip_type'] = flips.apply(flip_label, axis=1)

print(flips['flip_type'].value_counts())
flips[flips['flip_type'] != 'Consistent'].head(10)

flip_type
Consistent               493
Flipped in 2024            8
Flipped twice              6
Flipped in 2020, held      6
Name: count, dtype: int64


,fips,county,state,winner_2016,margin_2016,winner_2020,margin_2020,winner_2024,margin_2024,flip_type
7,04013,Maricopa County,Arizona,Republican,3.45,Democrat,-2.18,Republican,3.48,Flipped twice
19,13009,Baldwin County,Georgia,Democrat,-1.70,Democrat,-1.30,Republican,2.20,Flipped in 2024
31,13033,Burke County,Georgia,Democrat,-2.56,Republican,1.80,Republican,9.33,"Flipped in 2020, held"
95,13163,Jefferson County,Georgia,Democrat,-10.85,Democrat,-6.82,Republican,1.22,Flipped in 2024
164,13303,Washington County,Georgia,Democrat,-0.58,Democrat,-0.79,Republican,1.91,Flipped in 2024
214,26081,Kent County,Michigan,Republican,3.07,Democrat,-6.14,Democrat,-5.37,"Flipped in 2020, held"
218,26089,Leelanau County,Michigan,Republican,3.15,Democrat,-5.20,Democrat,-7.75,"Flipped in 2020, held"
234,26121,Muskegon County,Michigan,Democrat,-0.88,Democrat,-0.55,Republican,1.79,Flipped in 2024
246,26145,Saginaw County,Michigan,Republican,1.14,Democrat,-0.29,Republican,3.27,Flipped twice
260,37007,Anson County,North Carolina,Democrat,-12.56,Democrat,-4.18,Republican,2.50,Flipped in 2024


In [ ]:
# ── Chart: Flipped counties breakdown by state ────────────────────────────
flip_summary = (
    flips[flips['flip_type'] != 'Consistent']
    .groupby(['state', 'flip_type'])
    .size()
    .reset_index(name='count')
)

fig = px.bar(
    flip_summary,
    x='state',
    y='count',
    color='flip_type',
    barmode='group',
    title='Flipped Counties by State (Swing States Only)',
    labels={'count': 'Number of counties', 'state': '', 'flip_type': 'Flip Type'}
)
fig.update_layout(height=400)
fig.show()

---
## 7. Margin Shift Distribution Across Counties

This shows the *distribution* of county-level swings. If most counties shifted right, the histogram will skew positive.


In [ ]:
# Margin shift at county level: 2020 → 2024
county_margins = flips[['fips', 'county', 'state', 'margin_2016', 'margin_2020', 'margin_2024']].copy()
county_margins['shift_20_24'] = county_margins['margin_2024'] - county_margins['margin_2020']
county_margins['shift_16_24'] = county_margins['margin_2024'] - county_margins['margin_2016']

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['County Margin Shift: 2016→2020', 'County Margin Shift: 2020→2024'])

fig.add_trace(
    go.Histogram(x=county_margins['margin_2020'] - county_margins['margin_2016'],
                 nbinsx=40, name='2016→2020',
                 marker_color='#457B9D', opacity=0.75),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=county_margins['shift_20_24'],
                 nbinsx=40, name='2020→2024',
                 marker_color='#E63946', opacity=0.75),
    row=1, col=2
)

fig.update_layout(
    title='Distribution of County-Level Margin Shifts (Swing States)',
    height=400,
    xaxis_title='Shift in margin (pp, positive = toward R)',
    xaxis2_title='Shift in margin (pp, positive = toward R)'
)
fig.show()

---
## 8. Demographic Correlations

Now we bring in Census demographic data to ask: **does education, income, or racial composition correlate with how counties vote — or how they shifted?**

We'll use a pre-aggregated county demographics CSV from the Census ACS 5-year survey (2020).



In [ ]:
# Load county demographics — education, income, race (Census ACS)
# Source: USDA Economic Research Service / Census Bureau
demo_url = (
    "https://raw.githubusercontent.com/MEDSL/2018-elections-unoffical"
    "/master/election-context-2018.csv"
)

# Fallback: use a well-known county demographics CSV
demo_url_alt = (
    "https://raw.githubusercontent.com/evangambit/JsonOfCounties"
    "/master/counties.csv"
)

try:
    demo_raw = pd.read_csv(demo_url, encoding='latin1')
    print("Loaded election context data. Columns:", demo_raw.columns.tolist()[:15])
except Exception as e:
    print(f"Primary URL failed ({e}), trying fallback...")
    demo_raw = pd.read_csv(demo_url_alt)
    print("Loaded fallback. Columns:", demo_raw.columns.tolist()[:15])

Loaded election context data. Columns: ['state', 'county', 'fips', 'trump16', 'clinton16', 'otherpres16', 'romney12', 'obama12', 'otherpres12', 'demsen16', 'repsen16', 'othersen16', 'demhouse16', 'rephouse16', 'otherhouse16']


In [ ]:
# ── Adjust the column names below to match what loaded ─────────────────────
# Common column names in the MEDSL election context dataset:
#   fips, white_pct, black_pct, hispanic_pct, college_pct, median_hh_inc

# Standardize FIPS
demo_raw['fips'] = demo_raw['fips'].astype(str).str.zfill(5)

# Select demographic columns (rename to match if needed)
demo_cols = ['fips', 'white_pct', 'black_pct', 'hispanic_pct', 'college_pct', 'median_hh_inc']
demo_cols_present = [c for c in demo_cols if c in demo_raw.columns]
print("Available demo columns:", demo_cols_present)

demo = demo_raw[demo_cols_present].copy()

# Merge demographics with 2024 county results
df_2024_demo = (
    df_swing[df_swing['year'] == 2024]
    .merge(demo, on='fips', how='inner')
    .merge(county_margins[['fips', 'shift_20_24', 'shift_16_24']], on='fips', how='left')
)

print(f"\nMerged shape: {df_2024_demo.shape}")
df_2024_demo.head()

Available demo columns: ['fips', 'white_pct', 'black_pct', 'hispanic_pct', 'median_hh_inc']

Merged shape: (513, 17)


,fips,county,state,dem_votes,rep_votes,total_votes,year,dem_pct,rep_pct,margin,winner,white_pct,black_pct,hispanic_pct,median_hh_inc,shift_20_24,shift_16_24
0,04001,Apache County,Arizona,18872.00,12795.00,32019.00,2024,58.94,39.96,-18.98,Democrat,18.57,0.49,5.95,32460.00,14.70,17.90
1,04003,Cochise County,Arizona,22296.00,35936.00,58923.00,2024,37.84,60.99,23.15,Republican,56.30,3.71,34.40,45383.00,3.59,0.56
2,04005,Coconino County,Arizona,41504.00,27576.00,70067.00,2024,59.23,39.36,-19.88,Democrat,54.62,1.34,13.71,51106.00,4.18,-0.44
3,04007,Gila County,Arizona,8504.00,18901.00,27648.00,2024,30.76,68.36,37.60,Republican,63.22,0.55,18.55,40593.00,3.52,5.24
4,04009,Graham County,Arizona,3867.00,11177.00,15183.00,2024,25.47,73.62,48.15,Republican,51.46,1.81,32.10,47422.00,3.37,8.58


In [ ]:
# ── Chart: Income vs margin shift 2020→2024 ───────────────────────────────
if 'median_hh_inc' in df_2024_demo.columns and 'shift_20_24' in df_2024_demo.columns:
    fig = px.scatter(
        df_2024_demo.dropna(subset=['shift_20_24']),
        x='median_hh_inc',
        y='shift_20_24',
        color='state',
        size='total_votes',
        hover_data=['county'],
        trendline='ols',
        title='Median Household Income vs Margin Shift 2020→2024',
        labels={
            'median_hh_inc': 'Median Household Income ($)',
            'shift_20_24':   'Margin shift (pp, + = toward R)'
        },
        opacity=0.65
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(height=450)
    fig.show()

---
## 9. Correlation Heatmap

A quick summary of how all demographic variables correlate with vote margin and margin shift.


In [ ]:
corr_cols = [c for c in ['white_pct', 'black_pct', 'hispanic_pct',
                          'college_pct', 'median_hh_inc',
                          'margin', 'shift_20_24', 'shift_16_24']
             if c in df_2024_demo.columns]

corr_matrix = df_2024_demo[corr_cols].corr().round(2)

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation: Demographics vs Vote Margin / Shift'
)
fig.update_layout(height=450, width=650)
fig.show()